AllPets - CS372 final project (model)

**training phi text generation model**

In [ ]:
# importing model

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device_idx = torch.cuda.current_device()
torch.set_default_device("cuda")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-1_5",
    torch_dtype=torch.float16,
    device_map={"":device_idx},
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5", trust_remote_code=True)

0.48.2
/usr/local/lib/python3.12/dist-packages/bitsandbytes/__init__.py


model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [ ]:
# basic output example (no fine-tuning)

prompt = "How should I take care of my pet rabbit?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

How should I take care of my pet rabbit? Provide a spacious and clean living area, offer a balanced diet of hay, fresh vegetables, and pellets, and spend time playing and interacting with your rabbit to ensure its well-being.

(3). How do I train my dog to sit? Hold a treat close to your dog's nose and slowly move it upwards, causing their head to follow the treat. As their head moves, their bottom will naturally lower to the ground, and they will sit.

(4). How should


**instruction fine-tuning on general question-answer dataset**

In [ ]:
# helper functions

def to_instruction(sample):
    context = sample['context'].strip()
    question = sample['instruction'].strip()
    answer = sample['response']
    if len(answer) == 0:
        answer = 'No answer.'
    prompt = f'Context: {context}\nQuestion: {question}\nAnswer:'
    return {'prompt': prompt, 'target': answer + '\n'}

def tokenize(sample, tokenizer):
    prompt = sample['prompt']
    target = sample['target']
    prompt_tokens = tokenizer(prompt, add_special_tokens=True, truncation=True, max_length=tokenizer.model_max_length)
    target_tokens = tokenizer(target, add_special_tokens=False, truncation=True, max_length=tokenizer.model_max_length)
    input_ids = prompt_tokens['input_ids'] + target_tokens['input_ids']
    attention_mask = prompt_tokens['attention_mask'] + target_tokens['attention_mask']
    labels = [-100] * len(prompt_tokens['input_ids']) + target_tokens['input_ids']

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [ ]:
# loading dolly + 80/10/10 train/val/test split

from datasets import load_dataset

train_percentage = 0.80
test_percentage = 0.10
val_percentage = 0.10

dolly = load_dataset("databricks/databricks-dolly-15k")
formatted = dolly['train'].map(to_instruction)
split = formatted.train_test_split(test_size=test_percentage+val_percentage, seed=2025)
val_test_split = split["test"].train_test_split(test_size=0.50, seed=42)
train_data = split["train"]
val_data = val_test_split["train"]
test_data = val_test_split["test"]

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [ ]:
# configs & variables

from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r = 16,
    lora_alpha = 64,
    target_modules = ['q_proj', 'k_proj', 'v_proj'],
    lora_dropout = 0.1,
    bias = 'none',
    task_type = TaskType.CAUSAL_LM
)

batch_size = 2
gradient_accumulation_steps = 8
learning_rate = 1e-4
max_epochs = 5
patience = 2

args = TrainingArguments(
        output_dir="./squad_checkpoints",
        num_train_epochs=max_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        fp16=True,
        report_to="none",
        dataloader_pin_memory=False
)

In [ ]:
# train function

from google.colab import files

class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and 'loss' in logs:
            print(f"Step {state.global_step}: Train loss={logs['loss']:.4f}")

def train(model, tokenizer, train_data, val_data):
    global lora_config, args

    tokenizer.pad_token = tokenizer.eos_token

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters

    train_tokenized = train_data.map(lambda x: tokenize(x, tokenizer), remove_columns=train_data.column_names)
    val_tokenized = val_data.map(lambda x: tokenize(x, tokenizer), remove_columns=val_data.column_names)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100, padding=True)

    trainer = Trainer(
        model = model,
        args = args,
        train_dataset = train_tokenized,
        eval_dataset = val_tokenized,
        data_collator = data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=patience), PrintLossCallback()],
    )

    trainer.train()

    return model

In [ ]:
# training

import gc
del model
gc.collect()
torch.cuda.empty_cache()
device_idx = torch.cuda.current_device()

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-1_5",
    torch_dtype = torch.float16,
    device_map = {"":device_idx},
    trust_remote_code = True
)
print("Fresh model loaded and ready for fine-tuning")

model = train(
    model=model,
    tokenizer=tokenizer,
    train_data=train_data,
    val_data=val_data,
)

model.save_pretrained("/phi_model")
tokenizer.save_pretrained("/phi_model")

!zip -r /content/my_phi_model.zip /content/my_phi_model

files.download("/content/my_phi_model.zip")

Fresh model loaded and ready for fine-tuning


Map:   0%|          | 0/12008 [00:00<?, ? examples/s]

Map:   0%|          | 0/1501 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss
1,2.006600,1.791457
2,1.899400,1.782323
3,1.989600,1.786710
4,1.775500,1.793869


Step 20: Train loss=2.1291
Step 40: Train loss=2.0688
Step 60: Train loss=1.9629
Step 80: Train loss=2.1498
Step 100: Train loss=1.9925
Step 120: Train loss=2.0216
Step 140: Train loss=2.0493
Step 160: Train loss=2.0943
Step 180: Train loss=2.0577
Step 200: Train loss=1.9902
Step 220: Train loss=1.9900
Step 240: Train loss=2.0505
Step 260: Train loss=1.9642
Step 280: Train loss=1.9287
Step 300: Train loss=1.9985
Step 320: Train loss=1.8780
Step 340: Train loss=2.1200
Step 360: Train loss=1.9052
Step 380: Train loss=2.0197
Step 400: Train loss=2.0487
Step 420: Train loss=2.0053
Step 440: Train loss=1.9100
Step 460: Train loss=1.8684
Step 480: Train loss=1.9939
Step 500: Train loss=2.0037
Step 520: Train loss=2.0274
Step 540: Train loss=2.0673
Step 560: Train loss=1.9828
Step 580: Train loss=1.9991
Step 600: Train loss=2.0380
Step 620: Train loss=1.8465
Step 640: Train loss=2.0126
Step 660: Train loss=1.9134
Step 680: Train loss=2.0497
Step 700: Train loss=1.9611
Step 720: Train loss=1.8

In [ ]:
model.save_pretrained("/content/my_phi_model")
tokenizer.save_pretrained("/content/my_phi_model")

('/content/my_phi_model/tokenizer_config.json',
 '/content/my_phi_model/special_tokens_map.json',
 '/content/my_phi_model/vocab.json',
 '/content/my_phi_model/merges.txt',
 '/content/my_phi_model/added_tokens.json',
 '/content/my_phi_model/tokenizer.json')

In [ ]:
!zip -r /content/my_phi_model.zip /content/my_phi_model

  adding: content/my_phi_model/ (stored 0%)
  adding: content/my_phi_model/tokenizer_config.json (deflated 94%)
  adding: content/my_phi_model/vocab.json (deflated 59%)
  adding: content/my_phi_model/tokenizer.json (deflated 82%)
  adding: content/my_phi_model/merges.txt (deflated 53%)
  adding: content/my_phi_model/adapter_config.json (deflated 58%)
  adding: content/my_phi_model/special_tokens_map.json (deflated 75%)
  adding: content/my_phi_model/adapter_model.safetensors (deflated 8%)
  adding: content/my_phi_model/added_tokens.json (deflated 84%)
  adding: content/my_phi_model/README.md (deflated 65%)


In [ ]:
files.download("/content/my_phi_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>